In [6]:
from lib.kv import CFKV
from lib.aes import AES
import os
import json
import urllib

general_password = os.environ.get("GENERAL_PASSWORD")

kv = CFKV()
aes = AES(general_password)

pan_keys = aes.dec(kv.get("pan_keys"))
print(pan_keys)

{'expires_in': 2592000, 'refresh_token': '122.949d696a4ea8bd1cd818d25c5e5f3886.YlBtH8yUzI9RzsZcyn_d-oLpG8yMZPPW2cGcxoe.W6bmCQ', 'access_token': '121.6c4352e61b135e4cf677b86fae85ed94.YQJo-mRZLcLxhgd3KrWcOdh1updu8tz_0R2jsVS.bi_2NQ', 'session_secret': '', 'session_key': '', 'scope': 'basic netdisk'}


In [8]:
import requests
import os

class BaiduPan:
    def __init__(self, access_token):
        self.access_token = access_token
        self.api_base = "https://pan.baidu.com/rest/2.0/xpan/file"

    def list_files(self, path="/"):
        """列出目录下的文件和文件夹"""
        url = f"{self.api_base}?method=list&access_token={self.access_token}"
        params = {"dir": path, "web": 1, "num": 1000, "order": "name"}
        resp = requests.get(url, params=params)
        resp.raise_for_status()
        return resp.json()

    def create_folder(self, path):
        """创建目录"""
        url = f"{self.api_base}?method=create&access_token={self.access_token}"
        params = {"path": path, "isdir": 1}
        resp = requests.post(url, data=params)
        resp.raise_for_status()
        return resp.json()

    def delete(self, path):
        """删除文件或文件夹"""
        url = f"{self.api_base}?method=delete&access_token={self.access_token}"
        params = {"filelist": f'[{{"path":"{path}"}}]'}
        resp = requests.post(url, data=params)
        resp.raise_for_status()
        return resp.json()

    def upload(self, local_path, remote_path):
        """上传文件"""
        url = "https://pcs.baidu.com/rest/2.0/pcs/file"
        params = {
            "method": "upload",
            "access_token": self.access_token,
            "path": remote_path,
            "ondup": "overwrite"
        }
        with open(local_path, "rb") as f:
            files = {"file": f}
            resp = requests.post(url, params=params, files=files)
        resp.raise_for_status()
        return resp.json()

    def download(self, remote_path, local_path):
        """下载文件"""
        # 获取下载 URL
        url = f"https://pan.baidu.com/rest/2.0/xpan/file?method=download&access_token={self.access_token}&path={remote_path}"
        resp = requests.get(url, allow_redirects=True)
        resp.raise_for_status()
        with open(local_path, "wb") as f:
            f.write(resp.content)
        return local_path

# 使用示例
if __name__ == "__main__":
    ACCESS_TOKEN = pan_keys["access_token"]
    pan = BaiduPan(ACCESS_TOKEN)

    # 列文件
    print(pan.list_files("/data"))

    # 创建目录
    print(pan.create_folder("/data/test_folder"))

    # 上传文件
    print(pan.upload("./local_file.txt", "/data/test_folder/remote_file.txt"))

    # 下载文件
    print(pan.download("/data/test_folder/remote_file.txt", "./downloaded_file.txt"))

    # 删除文件
    print(pan.delete("/data/test_folder/remote_file.txt"))

{'errno': 0, 'guid_info': '', 'list': [{'tkbind_id': 0, 'server_filename': 'test_folder', 'category': 6, 'unlist': 0, 'isdir': 1, 'dir_empty': 1, 'oper_id': 3091442334, 'wpfile': 0, 'local_mtime': 1773405913, 'share': 0, 'pl': 2, 'local_ctime': 1773405913, 'empty': 0, 'real_category': '', 'black_tag': 0, 'extent_int2': 0, 'server_ctime': 1773405914, 'extent_tinyint7': 0, 'owner_id': 0, 'size': 0, 'extent_int8': 0, 'path': '/data/test_folder', 'from_type': 0, 'fs_id': 513298457421753, 'owner_type': 0, 'server_atime': 0, 'server_mtime': 1773405914, 'is_scene': 0}], 'request_id': 513860243874435109, 'guid': 0}
{'ctime': 1773406047, 'fs_id': 861644894193948, 'isdir': 1, 'mtime': 1773406047, 'path': '/data/test_folder_20260313_204726', 'status': 0, 'errno': 0, 'name': '/data/test_folder_20260313_204726', 'category': 6}


HTTPError: 403 Client Error: Forbidden for url: https://pcs.baidu.com/rest/2.0/pcs/file?method=upload&access_token=121.6c4352e61b135e4cf677b86fae85ed94.YQJo-mRZLcLxhgd3KrWcOdh1updu8tz_0R2jsVS.bi_2NQ&path=%2Fdata%2Ftest_folder%2Fremote_file.txt&ondup=overwrite